# 🛡 SentinAl — Unified Clinical AI for Neuro-Critical Care Risk Prediction

## Technical Report & Reproducible Code

---

| | |
|---|---|
| **Platform** | Privacy-First Edge AI (100% local inference) |
| **Modules** | PS2: Vital Signs Monitor \| PS1: Wound Grader \| PS5: Stroke Detector |
| **Stack** | PyTorch, Streamlit, EfficientNet-B0, Transformers, CLIP, Ollama |
| **Key Result** | AUROC 0.996 (Vitals) \| 97.05% Acc (Wounds) \| AUROC 0.982 (Stroke) |

---

> **SentinAl** (Sentinel + AI) is a unified clinical AI platform that predicts patient deterioration, grades diabetic foot wounds, and detects hemorrhagic strokes — all running 100% on-device with zero cloud data transfer.

## Table of Contents

1. [Problem Statements](#1-problem-statements)
2. [Environment Setup](#2-environment-setup)
3. [PS2: Vital Signs Deterioration Prediction](#3-ps2-vital-signs-deterioration-prediction)
    - 3.1 Data Preprocessing & Feature Engineering
    - 3.2 Model Architecture — Temporal Transformer
    - 3.3 Training Pipeline
    - 3.4 Evaluation & Results
4. [PS1: Diabetic Foot Wound Classification](#4-ps1-diabetic-foot-wound-classification)
    - 4.1 Dataset & Augmentation Strategy
    - 4.2 Model Architecture — EfficientNet-B0 + CLIP Validation
    - 4.3 Training Pipeline
    - 4.4 Evaluation & Results
5. [PS5: Brain Stroke Detection from CT Scans](#5-ps5-brain-stroke-detection-from-ct-scans)
    - 5.1 Dataset & Preprocessing
    - 5.2 Model Architecture — EfficientNet-B0
    - 5.3 Training Pipeline
    - 5.4 Evaluation & Results
6. [Specialist Recommender System](#6-specialist-recommender-system)
7. [Streamlit Application — Live Demo](#7-streamlit-application)
8. [Conclusions & Future Work](#8-conclusions--future-work)

---
<a id="1-problem-statements"></a>
# 1. Problem Statements

We address **three** critical challenges in neuro-critical care and diabetic foot management:

### Problem Statement 1 — AI-Based Prediction of High-Risk Plantar Pressure Zones for Prevention of Diabetic Foot Ulcers

> **Context:** Diabetic Foot Ulcers (DFUs) are one of the most severe complications of diabetes mellitus. Up to **80% of diabetes-related amputations** originate from non-healing foot ulcers.
>
> **Our Approach:** We built a **4-class Wagner Grade classifier** using EfficientNet-B0 (transfer learning from ImageNet) with CLIP-based image validation. The model classifies wound severity from Grade 1 (superficial) to Grade 4 (gangrene), providing immediate clinical action recommendations.
>
> **Deliverables:** Wound severity classification, confidence scores, probability distributions per grade, and specialist referral with urgency mapping.

### Problem Statement 2 — AI-Based Early Warning System for Patient Physiological Deterioration

> **Context:** Dependent elderly and neurological patients face silent physiological decline. Early warning signs appear through subtle changes in heart rate, respiratory rate, SpO2, blood pressure, and temperature — often **missed or detected too late**.
>
> **Our Approach:** We built a **Temporal Transformer** with 8-head self-attention that processes 12-hour sliding windows of 34 engineered clinical features to predict deterioration within the next 6-12 hours.
>
> **Deliverables:** Trained prediction model, data processing pipeline, 34 engineered features, Streamlit dashboard, and technical evaluation report.

### Problem Statement 5 — AI-Based Brain Stroke Detection and Lesion Segmentation from CT Scans

> **Context:** Every minute a stroke goes undetected, **1.9 million neurons die**. CT scans are the first diagnostic step, but correctly reading them is genuinely difficult — especially in lower-resource hospitals without experienced radiologists.
>
> **Our Approach:** We built an **EfficientNet-B0 binary classifier** for hemorrhagic stroke detection from brain CT scans, with an emergency alert system for positive findings.
>
> **Deliverables:** Trained classification model, data pipeline, Streamlit interface, and model evaluation report (Accuracy, F1, AUROC).

---
<a id="2-environment-setup"></a>
# 2. Environment Setup

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# Uncomment the line below if running on Google Colab
# !pip install torch torchvision streamlit plotly scikit-learn matplotlib transformers safetensors Pillow requests pandas numpy -q

In [ ]:
import os, sys, pickle, math, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    classification_report, confusion_matrix, roc_curve,
    precision_recall_curve, accuracy_score
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

# ── Device configuration ─────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥  Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"🔢  PyTorch: {torch.__version__}")
print(f"🐍  Python:  {sys.version.split()[0]}")
print(f"✅  Setup complete")

### Data Paths Configuration

> **Note:** Update these paths to match your local directory structure or Colab mount point. If running on Colab, upload your data to Google Drive and mount it.

In [ ]:
# ── Update these paths for your environment ──────────────────────────────────
# PS2 Data
PS2_TRAIN_CSV = "data/train.csv"           # Raw vital signs CSV
PS2_PROCESSED = "data/processed/"          # Output directory for processed numpy arrays
PS2_MODEL_DIR = "models/"                  # Saved model checkpoints

# PS1 Data
PS1_DATA_ROOT = "data/ps1/"               # Folder with train/valid subfolders containing Grade 1-4

# PS5 Data
PS5_DATA_ROOT = "data/ps5/"               # Folder with classification/train/val subfolders

# Create output directories
os.makedirs(PS2_PROCESSED, exist_ok=True)
os.makedirs(PS2_MODEL_DIR, exist_ok=True)
os.makedirs("outputs/", exist_ok=True)

print("📂 Directories configured")

---
<a id="3-ps2-vital-signs-deterioration-prediction"></a>
# 3. PS2: Vital Signs Deterioration Prediction

> **Task:** Predict whether a patient will deteriorate within the next 6-12 hours using continuous vital sign time-series data.
>
> **Approach:** Temporal Transformer with 8-head self-attention over 12-hour sliding windows of 34 engineered clinical features.

## 3.1 Data Preprocessing & Feature Engineering

Our preprocessing pipeline transforms raw vital sign CSV data into sliding-window tensor sequences:

1. **Patient ID Assignment** — Detect patient boundaries via `hour_from_admission` resets
2. **Feature Engineering** — Create 34 clinically meaningful derived features
3. **Categorical Encoding** — Label encode gender, admission type; ordinal encode oxygen device
4. **Standard Scaling** — Zero-mean, unit-variance normalization (fit on train, transform val)
5. **Sliding Windows** — 12-hour history with 1-hour step size

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PS2 PREPROCESSING PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

WINDOW = 12    # hours of history the model sees at once
STEP   = 1     # slide window by 1 hour each time

VITAL_COLS = [
    "heart_rate", "respiratory_rate", "spo2_pct", "temperature_c",
    "systolic_bp", "diastolic_bp", "oxygen_flow", "mobility_score",
    "nurse_alert", "wbc_count", "lactate", "creatinine",
    "crp_level", "hemoglobin", "sepsis_risk_score"
]
STATIC_COLS = ["age", "comorbidity_index"]
TARGET_COL  = "deterioration_next_12h"


def assign_patient_ids(df):
    """Detect patient boundaries by finding where hour_from_admission resets."""
    df = df.copy()
    df["patient_id"] = (df["hour_from_admission"].diff() < 0).cumsum()
    return df


def engineer_features(df):
    """
    Add 9 clinically meaningful derived features + 4 trend columns = 34 total features.

    Derived features:
    - Pulse pressure, MAP (Mean Arterial Pressure), Shock Index
    - Binary indicators: SpO2<94, tachycardia (HR>100), tachypnea (RR>20)
    - High lactate (>2.0), CRP high (>50)
    - qSOFA score (quick Sequential Organ Failure Assessment)
    - 4-hour rolling trends for HR, SpO2, RR, SBP
    """
    df = df.copy()

    # Hemodynamic features
    df["pulse_pressure"] = df["systolic_bp"] - df["diastolic_bp"]
    df["map"]            = df["diastolic_bp"] + df["pulse_pressure"] / 3
    df["shock_index"]    = df["heart_rate"] / (df["systolic_bp"] + 1e-6)

    # Binary threshold indicators
    df["spo2_below_94"]  = (df["spo2_pct"] < 94).astype(int)
    df["tachycardia"]    = (df["heart_rate"] > 100).astype(int)
    df["tachypnea"]      = (df["respiratory_rate"] > 20).astype(int)
    df["high_lactate"]   = (df["lactate"] > 2.0).astype(int)
    df["crp_high"]       = (df["crp_level"] > 50).astype(int)

    # qSOFA score (0-3)
    df["qsofa"] = (
        (df["respiratory_rate"] >= 22).astype(int) +
        (df["systolic_bp"] <= 100).astype(int) +
        df["nurse_alert"]
    )

    # 4-hour trends (captures trajectory, not just snapshots)
    for col in ["heart_rate", "spo2_pct", "respiratory_rate", "systolic_bp"]:
        df[f"{col}_trend4"] = df.groupby("patient_id")[col].transform(lambda x: x.diff(4))

    return df


def encode_categoricals(df, encoders=None):
    """Label encode gender & admission_type; ordinal encode oxygen_device."""
    df = df.copy()
    fit_mode = encoders is None
    if fit_mode:
        encoders = {}

    oxygen_order = {"none": 0, "nasal": 1, "mask": 2, "hfnc": 3, "niv": 4}
    df["oxygen_device_enc"] = df["oxygen_device"].map(oxygen_order).fillna(0)

    for col in ["gender", "admission_type"]:
        if fit_mode:
            le = LabelEncoder()
            df[f"{col}_enc"] = le.fit_transform(df[col].astype(str))
            encoders[col]    = le
        else:
            df[f"{col}_enc"] = encoders[col].transform(df[col].astype(str))

    return df, encoders


def build_windows(df, feature_cols, window=12, step=1, has_labels=True):
    """Create sliding windows of `window` hours with `step` hour stride."""
    X_seq, X_static, y_out, meta = [], [], [], []
    static_feats = STATIC_COLS + ["gender_enc", "admission_type_enc"]

    for pid, grp in df.groupby("patient_id", sort=False):
        grp = grp.reset_index(drop=True)
        n   = len(grp)
        if n < window:
            continue

        static_vec = grp[static_feats].iloc[0].values.astype(np.float32)

        for i in range(0, n - window + 1, step):
            seq = grp[feature_cols].iloc[i:i+window].values.astype(np.float32)
            X_seq.append(seq)
            X_static.append(static_vec)
            meta.append({"patient_id": pid, "window_end_hour": grp["hour_from_admission"].iloc[i+window-1]})
            if has_labels:
                y_out.append(int(grp[TARGET_COL].iloc[i+window-1]))

    X_seq    = np.array(X_seq,    dtype=np.float32)
    X_static = np.array(X_static, dtype=np.float32)
    y_out    = np.array(y_out,    dtype=np.float32) if has_labels else None
    return X_seq, X_static, y_out, meta


print("✅ PS2 preprocessing functions defined")
print(f"   Window size: {WINDOW} hours | Step: {STEP} hour")
print(f"   Raw vital columns: {len(VITAL_COLS)}")
print(f"   After engineering: 34 features (15 raw + 9 derived + 4 trends + 6 encoded)")

### Feature Engineering Summary

| Category | Features | Clinical Rationale |
|----------|----------|-------------------|
| **Hemodynamic** | Pulse Pressure, MAP, Shock Index | Core cardiovascular stability indicators |
| **Binary Thresholds** | SpO2<94, HR>100, RR>20, Lactate>2, CRP>50 | Standard clinical alarm thresholds |
| **Composite Score** | qSOFA (0-3) | Quick sepsis screening at bedside |
| **Temporal Trends** | 4-hr deltas for HR, SpO2, RR, SBP | Captures *trajectory* — is the patient improving or declining? |
| **Demographics** | Age, Gender, Comorbidity Index, Admission Type | Static risk factors |
| **Respiratory** | Oxygen flow, device type (ordinal 0-4) | Escalating O2 needs signal deterioration |

## 3.2 Model Architecture — Temporal Transformer

We implement two architectures and compare them:
1. **Temporal Transformer** (recommended) — 8-head self-attention for long-range temporal dependencies
2. **Bidirectional LSTM** — strong baseline for sequential data

### Why Transformer over LSTM?

| Aspect | LSTM | Transformer |
|--------|------|-------------|
| **Temporal range** | Sequential — information dilutes over long sequences | Direct attention between any 2 hours |
| **Parallelism** | Sequential processing | Fully parallel — faster GPU utilization |
| **Interpretability** | Hidden state (black box) | Attention weights show *which hours matter* |
| **Performance** | Strong baseline | Higher AUROC in our experiments |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PS2 MODEL ARCHITECTURES
# ══════════════════════════════════════════════════════════════════════════════

class BidirectionalLSTM(nn.Module):
    """Bidirectional LSTM baseline for vital sign deterioration prediction."""

    def __init__(self, temporal_input_size, static_input_size,
                 hidden_size=128, num_layers=2, dropout=0.35):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=temporal_input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.static_encoder = nn.Sequential(
            nn.Linear(static_input_size, 32), nn.ReLU(), nn.Dropout(0.2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2 + 32, 128), nn.LayerNorm(128), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout * 0.5),
            nn.Linear(64, 1),
        )

    def forward(self, x_seq, x_static):
        lstm_out, _ = self.lstm(x_seq)
        last_step   = lstm_out[:, -1, :]
        static_emb  = self.static_encoder(x_static)
        fused       = torch.cat([last_step, static_emb], dim=1)
        return torch.sigmoid(self.classifier(fused)).squeeze(1)


class TemporalTransformer(nn.Module):
    """
    Temporal Transformer for vital sign deterioration prediction.

    Architecture:
    1. Input projection: temporal_features → d_model (128)
    2. Positional embedding: learned embeddings for temporal positions
    3. Transformer encoder: 3 layers × 8 attention heads × 256 FFN
    4. Attention pooling: softmax-weighted aggregation over time
    5. Static encoder: demographics → 32-dim embedding
    6. Classifier: fused [128+32] → 128 → 1 (sigmoid)
    """

    def __init__(self, temporal_input_size, static_input_size,
                 d_model=128, nhead=8, num_encoder_layers=3,
                 dim_feedforward=256, dropout=0.2, max_seq_len=72):
        super().__init__()
        self.input_proj    = nn.Linear(temporal_input_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)

        self.static_encoder = nn.Sequential(
            nn.Linear(static_input_size, 64), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, 32),
        )
        self.attn_pool  = nn.Linear(d_model, 1)
        self.classifier = nn.Sequential(
            nn.Linear(d_model + 32, 128), nn.LayerNorm(128), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(128, 1),
        )

    def forward(self, x_seq, x_static):
        B, T, _ = x_seq.shape
        x        = self.input_proj(x_seq)
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        x        = x + self.pos_embedding(positions)
        enc      = self.transformer(x)
        weights  = torch.softmax(self.attn_pool(enc), dim=1)
        pooled   = (enc * weights).sum(dim=1)
        static_emb = self.static_encoder(x_static)
        fused    = torch.cat([pooled, static_emb], dim=1)
        return torch.sigmoid(self.classifier(fused)).squeeze(1)


# ── Model summary ────────────────────────────────────────────────────────────
print("=" * 60)
print("PS2 MODEL ARCHITECTURES")
print("=" * 60)

for name, ModelClass in [("Temporal Transformer", TemporalTransformer),
                          ("Bidirectional LSTM", BidirectionalLSTM)]:
    model = ModelClass(temporal_input_size=34, static_input_size=4)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{name}:")
    print(f"  Total parameters    : {total:,}")
    print(f"  Trainable parameters: {trainable:,}")

    # Test forward pass
    dummy_seq    = torch.randn(2, 12, 34)
    dummy_static = torch.randn(2, 4)
    out = model(dummy_seq, dummy_static)
    print(f"  Input:  seq={dummy_seq.shape}, static={dummy_static.shape}")
    print(f"  Output: {out.shape} → probabilities: {out.detach().numpy()}")

del model
print("\n✅ Both architectures verified")

## 3.3 Training Pipeline

### Class Imbalance Strategy

Our dataset has a **94.6% stable vs 5.4% deteriorating** class distribution — a severe 17:1 imbalance. We address this with three complementary techniques:

1. **Focal Loss** (α=0.75, γ=2.0) — Downweights easy-to-classify stable patients, focuses learning on hard-to-detect deterioration cases
2. **WeightedRandomSampler** — Oversamples minority class during training for balanced mini-batches
3. **Threshold Optimization** — Instead of default 0.5, we sweep 200 thresholds and pick the one maximizing F1-score (optimal: 0.841)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PS2 TRAINING PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

class FocalLoss(nn.Module):
    """
    Focal Loss for class imbalance (Lin et al., 2017).

    Adds a modulating factor (1 - p_t)^γ to cross-entropy:
    - Easy examples (p_t → 1): weight → 0 (effectively ignored)
    - Hard examples (p_t → 0): weight → 1 (full contribution)

    α balances positive/negative classes.
    γ controls focus strength (higher = more focus on hard examples).
    """
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, probs, targets):
        bce          = F.binary_cross_entropy(probs, targets, reduction="none")
        p_t          = probs * targets + (1 - probs) * (1 - targets)
        alpha_t      = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        return (focal_weight * bce).mean()


# ── Training configuration ───────────────────────────────────────────────────
PS2_CONFIG = {
    "batch_size":   256,
    "epochs":       60,
    "lr":           3e-4,
    "weight_decay": 1e-4,
    "val_split":    0.15,
    "patience":     8,
    "focal_alpha":  0.75,
    "focal_gamma":  2.0,
    "grad_clip":    1.0,
    "scheduler":    "CosineAnnealingLR",
    "optimizer":    "AdamW",
}

print("PS2 Training Configuration:")
print("-" * 40)
for k, v in PS2_CONFIG.items():
    print(f"  {k:20s}: {v}")


def make_loader(X_seq, X_static, y, batch_size=256, use_sampler=False, shuffle=True):
    """Create DataLoader with optional WeightedRandomSampler for class balancing."""
    ds = TensorDataset(
        torch.FloatTensor(X_seq),
        torch.FloatTensor(X_static),
        torch.FloatTensor(y),
    )
    sampler = None
    if use_sampler:
        counts  = np.bincount(y.astype(int))
        weights = 1.0 / counts[y.astype(int)]
        sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
        shuffle = False
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      sampler=sampler, num_workers=0)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_seq, X_stat, y_batch in loader:
        X_seq, X_stat, y_batch = X_seq.to(device), X_stat.to(device), y_batch.to(device)
        preds = model(X_seq, X_stat)
        loss  = criterion(preds, y_batch)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), PS2_CONFIG["grad_clip"])
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate_ps2(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for X_seq, X_stat, y_batch in loader:
        probs = model(X_seq.to(device), X_stat.to(device)).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(y_batch.numpy())
    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    auroc  = roc_auc_score(labels, probs)
    auprc  = average_precision_score(labels, probs)
    return auroc, auprc, probs, labels


def train_ps2(X_seq, X_static, y, model_name="transformer"):
    """Full PS2 training loop with early stopping and threshold optimization."""

    # Train/val split
    idx = np.arange(len(y))
    tr_idx, vl_idx = train_test_split(idx, test_size=PS2_CONFIG["val_split"],
                                       stratify=y, random_state=SEED)

    print(f"Train: {len(tr_idx):,} windows | Val: {len(vl_idx):,} windows")
    print(f"Positive rate — Train: {y[tr_idx].mean()*100:.2f}% | Val: {y[vl_idx].mean()*100:.2f}%")

    # Create model
    temporal_dim = X_seq.shape[2]
    static_dim   = X_static.shape[1]

    if model_name == "transformer":
        model = TemporalTransformer(temporal_dim, static_dim).to(DEVICE)
    else:
        model = BidirectionalLSTM(temporal_dim, static_dim).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {model_name} | Parameters: {total_params:,}")

    # Loaders
    tr_loader = make_loader(X_seq[tr_idx], X_static[tr_idx], y[tr_idx],
                            batch_size=PS2_CONFIG["batch_size"], use_sampler=True)
    vl_loader = make_loader(X_seq[vl_idx], X_static[vl_idx], y[vl_idx],
                            batch_size=PS2_CONFIG["batch_size"], shuffle=False)

    # Training setup
    criterion = FocalLoss(alpha=PS2_CONFIG["focal_alpha"], gamma=PS2_CONFIG["focal_gamma"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=PS2_CONFIG["lr"],
                                   weight_decay=PS2_CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=PS2_CONFIG["epochs"], eta_min=1e-6)

    # Training loop
    save_path = os.path.join(PS2_MODEL_DIR, f"best_{model_name}.pt")
    best_auroc = 0.0
    patience_cnt = 0
    history = {"train_loss": [], "val_auroc": [], "val_auprc": []}

    print(f"\n{'Epoch':>5} | {'Train Loss':>10} | {'Val AUROC':>9} | {'Val AUPRC':>9} | {'Status':>8}")
    print("-" * 52)

    for epoch in range(1, PS2_CONFIG["epochs"] + 1):
        train_loss = train_epoch(model, tr_loader, optimizer, criterion, DEVICE)
        val_auroc, val_auprc, _, _ = evaluate_ps2(model, vl_loader, DEVICE)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_auroc"].append(val_auroc)
        history["val_auprc"].append(val_auprc)

        improved = val_auroc > best_auroc
        if improved:
            best_auroc = val_auroc
            patience_cnt = 0
            torch.save(model.state_dict(), save_path)

        print(f"{epoch:>5} | {train_loss:>10.4f} | {val_auroc:>9.4f} | "
              f"{val_auprc:>9.4f} | {'* BEST' if improved else ''}")

        if not improved:
            patience_cnt += 1
            if patience_cnt >= PS2_CONFIG["patience"]:
                print(f"\nEarly stopping at epoch {epoch}. Best AUROC: {best_auroc:.4f}")
                break

    # Threshold optimization
    model.load_state_dict(torch.load(save_path, map_location=DEVICE, weights_only=True))
    _, _, probs, labels = evaluate_ps2(model, vl_loader, DEVICE)

    best_t, best_f1 = 0.5, 0
    for t in np.linspace(0.05, 0.95, 200):
        f1 = f1_score(labels, (probs >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t

    with open(os.path.join(PS2_MODEL_DIR, f"threshold_{model_name}.txt"), "w") as f:
        f.write(str(best_t))

    print(f"\n{'='*52}")
    print(f"Best AUROC         : {best_auroc:.4f}")
    print(f"Optimal Threshold  : {best_t:.4f} (F1={best_f1:.4f})")
    print(f"Model saved to     : {save_path}")

    return model, history, best_t, probs, labels


print("\n✅ PS2 training pipeline defined")

## 3.4 Evaluation & Results

### PS2 Performance Metrics

| Metric | Value |
|--------|-------|
| **AUROC** | **0.9960** |
| **AUPRC** | High |
| **Sensitivity** | **88.9%** |
| **Specificity** | High |
| **Optimal Threshold** | 0.841 |
| **Training Windows** | 216,248 |
| **Patients** | ~7,000 |
| **Positive Rate** | 5.4% |
| **Best Epoch** | 51 / 60 |
| **Total Parameters** | 434,146 |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PS2 EVALUATION & VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════════

def plot_ps2_evaluation(probs, labels, threshold, history=None):
    """Generate comprehensive evaluation plots for PS2."""

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("PS2: Vital Signs Deterioration Predictor — Evaluation Report",
                 fontsize=16, fontweight="bold", y=1.02)

    # 1. ROC Curve
    fpr, tpr, _ = roc_curve(labels, probs)
    auroc = roc_auc_score(labels, probs)
    axes[0, 0].plot(fpr, tpr, color="#0EA5E9", lw=2.5, label=f"AUROC = {auroc:.4f}")
    axes[0, 0].plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.5)
    axes[0, 0].fill_between(fpr, tpr, alpha=0.1, color="#0EA5E9")
    axes[0, 0].set_xlabel("False Positive Rate")
    axes[0, 0].set_ylabel("True Positive Rate")
    axes[0, 0].set_title("ROC Curve")
    axes[0, 0].legend(loc="lower right", fontsize=12)
    axes[0, 0].grid(alpha=0.3)

    # 2. Precision-Recall Curve
    prec, rec, _ = precision_recall_curve(labels, probs)
    auprc = average_precision_score(labels, probs)
    axes[0, 1].plot(rec, prec, color="#10B981", lw=2.5, label=f"AUPRC = {auprc:.4f}")
    axes[0, 1].axhline(labels.mean(), color="gray", ls="--", lw=0.8, label="Baseline")
    axes[0, 1].fill_between(rec, prec, alpha=0.1, color="#10B981")
    axes[0, 1].set_xlabel("Recall")
    axes[0, 1].set_ylabel("Precision")
    axes[0, 1].set_title("Precision-Recall Curve")
    axes[0, 1].legend(fontsize=11)
    axes[0, 1].grid(alpha=0.3)

    # 3. Score Distribution
    y_pred = (probs >= threshold).astype(int)
    axes[0, 2].hist(probs[labels == 0], bins=50, alpha=0.7, color="#94A3B8", label="Stable", density=True)
    axes[0, 2].hist(probs[labels == 1], bins=50, alpha=0.7, color="#EF4444", label="Deteriorating", density=True)
    axes[0, 2].axvline(threshold, color="black", ls="--", lw=2, label=f"Threshold={threshold:.3f}")
    axes[0, 2].set_xlabel("Predicted Probability")
    axes[0, 2].set_ylabel("Density")
    axes[0, 2].set_title("Score Distribution")
    axes[0, 2].legend(fontsize=10)
    axes[0, 2].grid(alpha=0.3)

    # 4. Confusion Matrix
    cm = confusion_matrix(labels, y_pred)
    im = axes[1, 0].imshow(cm, cmap="Blues", aspect="auto")
    for i in range(2):
        for j in range(2):
            axes[1, 0].text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=14,
                           color="white" if cm[i, j] > cm.max() / 2 else "black")
    axes[1, 0].set_xticks([0, 1]); axes[1, 0].set_xticklabels(["Stable", "Deteriorating"])
    axes[1, 0].set_yticks([0, 1]); axes[1, 0].set_yticklabels(["Stable", "Deteriorating"])
    axes[1, 0].set_xlabel("Predicted"); axes[1, 0].set_ylabel("Actual")
    axes[1, 0].set_title("Confusion Matrix")

    # 5. F1 vs Threshold
    thresholds = np.linspace(0.05, 0.95, 200)
    f1_scores = [f1_score(labels, (probs >= t).astype(int), zero_division=0) for t in thresholds]
    axes[1, 1].plot(thresholds, f1_scores, color="#F59E0B", lw=2.5)
    axes[1, 1].axvline(threshold, color="black", ls="--", lw=2, label=f"Optimal={threshold:.3f}")
    best_f1_idx = np.argmax(f1_scores)
    axes[1, 1].scatter([thresholds[best_f1_idx]], [f1_scores[best_f1_idx]],
                       color="red", s=100, zorder=5, label=f"Best F1={f1_scores[best_f1_idx]:.3f}")
    axes[1, 1].set_xlabel("Threshold")
    axes[1, 1].set_ylabel("F1 Score")
    axes[1, 1].set_title("F1 Score vs Threshold")
    axes[1, 1].legend(fontsize=10)
    axes[1, 1].grid(alpha=0.3)

    # 6. Training History (or metrics summary)
    if history:
        ax2 = axes[1, 2].twinx()
        axes[1, 2].plot(history["train_loss"], color="#EF4444", lw=2, label="Train Loss")
        ax2.plot(history["val_auroc"], color="#0EA5E9", lw=2, label="Val AUROC")
        axes[1, 2].set_xlabel("Epoch")
        axes[1, 2].set_ylabel("Loss", color="#EF4444")
        ax2.set_ylabel("AUROC", color="#0EA5E9")
        axes[1, 2].set_title("Training History")
        lines1, labels1 = axes[1, 2].get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        axes[1, 2].legend(lines1 + lines2, labels1 + labels2, fontsize=10)
        axes[1, 2].grid(alpha=0.3)
    else:
        # Summary text
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        summary = (
            f"AUROC:       {auroc:.4f}\n"
            f"AUPRC:       {auprc:.4f}\n"
            f"Sensitivity: {sens:.4f}\n"
            f"Specificity: {spec:.4f}\n"
            f"Threshold:   {threshold:.4f}\n"
            f"Samples:     {len(labels):,}\n"
            f"Positive:    {labels.sum():.0f} ({labels.mean()*100:.1f}%)"
        )
        axes[1, 2].text(0.1, 0.5, summary, transform=axes[1, 2].transAxes,
                        fontsize=13, family="monospace", va="center",
                        bbox=dict(boxstyle="round,pad=0.5", facecolor="#F1F5F9"))
        axes[1, 2].set_title("Performance Summary")
        axes[1, 2].axis("off")

    plt.tight_layout()
    plt.savefig("outputs/ps2_evaluation.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("📊 Saved to outputs/ps2_evaluation.png")


# ── Print final classification report ────────────────────────────────────────
def print_ps2_report(probs, labels, threshold):
    y_pred = (probs >= threshold).astype(int)
    print("\n" + "=" * 60)
    print("PS2 FINAL EVALUATION REPORT")
    print("=" * 60)
    print(f"AUROC          : {roc_auc_score(labels, probs):.4f}")
    print(f"AUPRC          : {average_precision_score(labels, probs):.4f}")
    print(f"Threshold      : {threshold:.4f}")
    print(f"\n{classification_report(labels, y_pred, target_names=['Stable', 'Deteriorating'])}")
    cm = confusion_matrix(labels, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"Sensitivity    : {tp/(tp+fn):.4f}")
    print(f"Specificity    : {tn/(tn+fp):.4f}")
    print("=" * 60)


print("✅ PS2 evaluation functions defined")

### Run PS2 Training

> **Note:** Training requires the PS2 dataset (`train.csv`). If the data is not available, the cell below will be skipped. The reported metrics are from our completed training run.

> **Reported Results (from completed training):**
> - **AUROC: 0.9960** | Sensitivity: 88.9% | Threshold: 0.841
> - Training: 293K rows, 7K patients, 216K sliding windows
> - Class balance: 94.6% stable vs 5.4% deteriorating

In [ ]:
# ── Run PS2 training (uncomment to execute) ──────────────────────────────────
# This cell runs the full training pipeline. It requires:
#   1. PS2_TRAIN_CSV pointing to a valid train.csv
#   2. ~10-20 minutes on GPU, ~1-2 hours on CPU

# Uncomment below to run:
# ─────────────────────────────────────────────────────────────────────────────
# print("Loading and preprocessing PS2 data...")
# train_df = pd.read_csv(PS2_TRAIN_CSV)
# train_df = assign_patient_ids(train_df)
# train_df = engineer_features(train_df)
# train_df, encoders = encode_categoricals(train_df)
#
# trend_cols   = [f"{c}_trend4" for c in ["heart_rate","spo2_pct","respiratory_rate","systolic_bp"]]
# engineered   = ["pulse_pressure","map","shock_index","spo2_below_94",
#                  "tachycardia","tachypnea","high_lactate","crp_high","qsofa"]
# feature_cols = VITAL_COLS + engineered + trend_cols + ["oxygen_device_enc","hour_from_admission"]
#
# for col in trend_cols:
#     train_df[col] = train_df.groupby("patient_id")[col].transform(lambda x: x.fillna(0))
#
# scaler = StandardScaler()
# train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
#
# X_seq, X_static, y, _ = build_windows(train_df, feature_cols, WINDOW, STEP, has_labels=True)
# print(f"Windows: {X_seq.shape} | Positive rate: {y.mean()*100:.2f}%")
#
# model, history, threshold, probs, labels = train_ps2(X_seq, X_static, y, model_name="transformer")
# plot_ps2_evaluation(probs, labels, threshold, history)
# print_ps2_report(probs, labels, threshold)

print("💡 PS2 training cell ready — uncomment to run")
print("   Reported results: AUROC=0.9960 | Sensitivity=88.9% | Threshold=0.841")

---
<a id="4-ps1-diabetic-foot-wound-classification"></a>
# 4. PS1: Diabetic Foot Wound Classification

> **Task:** Classify diabetic foot wound images into 4 Wagner grades (Grade 1–4).
>
> **Approach:** EfficientNet-B0 (transfer learning from ImageNet) with CLIP-based image validation to reject non-medical uploads.

## 4.1 Dataset & Augmentation Strategy

| Property | Value |
|----------|-------|
| **Total images** | 9,934 |
| **Classes** | 4 (Wagner Grade 1, 2, 3, 4) |
| **Split** | 85% train / 15% val (stratified) |
| **Key fix** | Combined train+valid folders and re-split — original valid had only 40 Grade 3 images |
| **Augmentation** | Random crop, flips, rotation (20°), ColorJitter, 5% grayscale |
| **Normalization** | ImageNet mean/std |

### Wagner Grading System

| Grade | Description | Clinical Action |
|-------|-------------|----------------|
| Grade 1 | Superficial ulcer | Routine podiatry |
| Grade 2 | Deep ulcer (tendon/bone) | Diabetology consult within a week |
| Grade 3 | Abscess / Osteomyelitis | Urgent surgical review within 24h |
| Grade 4 | Gangrene | Emergency — immediate intervention |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PS1 DATASET & AUGMENTATION
# ══════════════════════════════════════════════════════════════════════════════

GRADE_TO_LABEL = {"Grade 1": 0, "Grade 2": 1, "Grade 3": 2, "Grade 4": 3}
LABEL_TO_GRADE = {v: k for k, v in GRADE_TO_LABEL.items()}

IMG_SIZE      = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Augmentation transforms ──────────────────────────────────────────────────
ps1_train_transform = T.Compose([
    T.Resize((256, 256)),
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    T.RandomGrayscale(p=0.05),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

ps1_val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class FootWoundDataset(Dataset):
    """PyTorch Dataset for foot wound images."""
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)


def collect_ps1_samples(root_dir):
    """Collect all images from train + valid folders for re-splitting."""
    all_samples = []
    for split_folder in ["train", "valid"]:
        split_dir = os.path.join(root_dir, split_folder)
        if not os.path.exists(split_dir):
            continue
        for grade_name, label in GRADE_TO_LABEL.items():
            grade_dir = os.path.join(split_dir, grade_name)
            if not os.path.exists(grade_dir):
                continue
            for fname in os.listdir(grade_dir):
                if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                    all_samples.append((os.path.join(grade_dir, fname), label))
    return all_samples


def get_ps1_dataloaders(root_dir, batch_size=64, val_size=0.15):
    """Create train/val DataLoaders with stratified split and class weights."""
    all_samples = collect_ps1_samples(root_dir)
    all_labels  = [label for _, label in all_samples]

    print(f"Total images: {len(all_samples)}")
    for grade_name, label in GRADE_TO_LABEL.items():
        print(f"  {grade_name}: {all_labels.count(label)}")

    train_samples, val_samples = train_test_split(
        all_samples, test_size=val_size, stratify=all_labels, random_state=SEED)

    print(f"\nTrain: {len(train_samples)} | Val: {len(val_samples)}")

    # Compute class weights for imbalanced loss
    train_labels = [l for _, l in train_samples]
    total = len(train_labels)
    class_weights = torch.FloatTensor([total / (4 * train_labels.count(i)) for i in range(4)])
    print(f"Class weights: {[f'{w:.3f}' for w in class_weights]}")

    train_loader = DataLoader(FootWoundDataset(train_samples, ps1_train_transform),
                              batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(FootWoundDataset(val_samples, ps1_val_transform),
                              batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

    return train_loader, val_loader, class_weights


print("✅ PS1 dataset pipeline defined")

## 4.2 Model Architecture — EfficientNet-B0 + CLIP Validation

### Transfer Learning Strategy
- **Backbone:** EfficientNet-B0 pretrained on 1M ImageNet images
- **Custom Head:** 1280 → Dropout(0.5) → 256 → ReLU → Dropout(0.25) → 4
- **All layers trainable** (full fine-tuning, not frozen backbone)

### CLIP Image Validation
Before inference, we use OpenAI's **CLIP** model in zero-shot mode to verify the uploaded image is actually a foot wound:
- 8 candidate labels: foot wound, healthy foot, animal, food, landscape, portrait, screenshot, random object
- Reject if "foot wound" confidence < 25%
- Prevents garbage-in-garbage-out

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PS1 MODEL ARCHITECTURE — EfficientNet-B0
# ══════════════════════════════════════════════════════════════════════════════

NUM_CLASSES_PS1 = 4  # Wagner Grade 1-4

class FootWoundClassifier(nn.Module):
    """
    EfficientNet-B0 fine-tuned for 4-class foot wound grading.

    Architecture:
        Input image (224×224×3)
            ↓
        EfficientNet-B0 backbone (pretrained — knows edges, textures, shapes)
            ↓
        Dropout (0.5) → Linear (1280→256) → ReLU → Dropout (0.25)
            ↓
        Linear (256→4)  ← scores for each Wagner grade
    """
    def __init__(self, num_classes=NUM_CLASSES_PS1, dropout=0.5):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_features = self.backbone.classifier[1].in_features  # 1280
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)


# ── Model summary ────────────────────────────────────────────────────────────
model_ps1 = FootWoundClassifier()
total = sum(p.numel() for p in model_ps1.parameters())
trainable = sum(p.numel() for p in model_ps1.parameters() if p.requires_grad)
print(f"FootWoundClassifier (EfficientNet-B0):")
print(f"  Total parameters    : {total:,}")
print(f"  Trainable parameters: {trainable:,}")

# Test forward pass
dummy = torch.randn(2, 3, 224, 224)
out = model_ps1(dummy)
print(f"  Input:  {dummy.shape}")
print(f"  Output: {out.shape} → 4 class logits")
print(f"  Softmax: {torch.softmax(out, dim=1).detach().numpy().round(3)}")

del model_ps1
print("\n✅ PS1 model verified")

## 4.3 PS1 Training Pipeline

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PS1 TRAINING PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

PS1_CONFIG = {
    "epochs":       30,
    "batch_size":   64,
    "lr":           2e-4,
    "weight_decay": 5e-4,
    "patience":     8,
    "grad_clip":    1.0,
    "scheduler":    "CosineAnnealingLR",
    "loss":         "Weighted CrossEntropyLoss",
}

print("PS1 Training Configuration:")
print("-" * 40)
for k, v in PS1_CONFIG.items():
    print(f"  {k:20s}: {v}")


def train_ps1(root_dir, batch_size=64, epochs=30, patience=8):
    """Full PS1 training loop."""
    train_loader, val_loader, class_weights = get_ps1_dataloaders(root_dir, batch_size)

    model = FootWoundClassifier().to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=PS1_CONFIG["lr"],
                                   weight_decay=PS1_CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    save_path = os.path.join(PS2_MODEL_DIR, "best_ps1.pt")
    best_val_acc, patience_cnt = 0.0, 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    print(f"\n{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7} | {'Note':>10}")
    print("-" * 62)

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc  = correct / total * 100

        # Validate
        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                val_loss_sum += criterion(outputs, labels).item()
                preds = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss = val_loss_sum / len(val_loader)
        val_acc  = val_correct / val_total * 100
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            patience_cnt = 0
            torch.save({"epoch": epoch, "model_state": model.state_dict(), "val_acc": val_acc}, save_path)

        note = f"* BEST" if improved else ""
        print(f"{epoch:>5} | {train_loss:>10.4f} | {train_acc:>8.2f}% | {val_loss:>8.4f} | {val_acc:>6.2f}% | {note}")

        if not improved:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f"\nEarly stopping at epoch {epoch}")
                break

    # Final evaluation
    checkpoint = torch.load(save_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    print(f"\nBest validation accuracy: {best_val_acc:.2f}%")
    print(f"\n{classification_report(all_labels, all_preds, target_names=[LABEL_TO_GRADE[i] for i in range(4)])}")

    return model, history, all_preds, all_labels


print("\n✅ PS1 training pipeline defined")

## 4.4 PS1 Evaluation & Results

### PS1 Performance Metrics

| Metric | Value |
|--------|-------|
| **Validation Accuracy** | **97.05%** |
| **F1 Macro** | **0.970** |
| **Total Images** | 9,934 |
| **Training Epochs** | 27 (early stopped at 30) |
| **Key Fix** | Grade 3 augmentation (40→800+ images) |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PS1 EVALUATION VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════════

def plot_ps1_evaluation(preds, labels, history=None):
    """Generate evaluation plots for PS1."""
    grade_names = [LABEL_TO_GRADE[i] for i in range(4)]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("PS1: Diabetic Foot Wound Grader — Evaluation Report",
                 fontsize=15, fontweight="bold")

    # 1. Confusion Matrix
    cm = confusion_matrix(labels, preds)
    colors = ["#22C55E", "#F59E0B", "#F97316", "#EF4444"]
    im = axes[0].imshow(cm, cmap="OrRd", aspect="auto")
    for i in range(4):
        for j in range(4):
            axes[0].text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=13,
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
    axes[0].set_xticks(range(4)); axes[0].set_xticklabels(grade_names, rotation=45)
    axes[0].set_yticks(range(4)); axes[0].set_yticklabels(grade_names)
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
    axes[0].set_title("Confusion Matrix")

    # 2. Per-class metrics
    report = classification_report(labels, preds, target_names=grade_names, output_dict=True)
    grades = grade_names
    f1s     = [report[g]["f1-score"] for g in grades]
    precs   = [report[g]["precision"] for g in grades]
    recalls = [report[g]["recall"] for g in grades]

    x = np.arange(4)
    w = 0.25
    axes[1].bar(x - w, precs, w, label="Precision", color="#0EA5E9", alpha=0.8)
    axes[1].bar(x,     recalls, w, label="Recall", color="#10B981", alpha=0.8)
    axes[1].bar(x + w, f1s, w, label="F1", color="#F59E0B", alpha=0.8)
    axes[1].set_xticks(x); axes[1].set_xticklabels(grade_names)
    axes[1].set_ylim(0, 1.1); axes[1].set_ylabel("Score")
    axes[1].set_title("Per-Class Metrics")
    axes[1].legend()
    axes[1].grid(axis="y", alpha=0.3)

    # 3. Training history or class distribution
    if history:
        ax2 = axes[2].twinx()
        axes[2].plot(history["train_loss"], color="#EF4444", lw=2, label="Train Loss")
        axes[2].plot(history["val_loss"], color="#F59E0B", lw=2, ls="--", label="Val Loss")
        ax2.plot(history["val_acc"], color="#0EA5E9", lw=2, label="Val Acc")
        axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Loss")
        ax2.set_ylabel("Accuracy (%)")
        axes[2].set_title("Training History")
        lines1, labels1 = axes[2].get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        axes[2].legend(lines1 + lines2, labels1 + labels2, fontsize=9)
    else:
        counts = [list(labels).count(i) for i in range(4)]
        axes[2].bar(grade_names, counts, color=colors, alpha=0.8, edgecolor="white", lw=2)
        axes[2].set_ylabel("Count"); axes[2].set_title("Class Distribution (Validation)")
        for i, c in enumerate(counts):
            axes[2].text(i, c + 5, str(c), ha="center", fontweight="bold")

    plt.tight_layout()
    plt.savefig("outputs/ps1_evaluation.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("📊 Saved to outputs/ps1_evaluation.png")


# ── Run PS1 training (uncomment to execute) ──────────────────────────────────
# model_ps1, hist_ps1, preds_ps1, labels_ps1 = train_ps1(PS1_DATA_ROOT)
# plot_ps1_evaluation(preds_ps1, labels_ps1, hist_ps1)

print("💡 PS1 training cell ready — uncomment to run")
print("   Reported results: Accuracy=97.05% | F1 Macro=0.970")

---
<a id="5-ps5-brain-stroke-detection-from-ct-scans"></a>
# 5. PS5: Brain Stroke Detection from CT Scans

> **Task:** Detect hemorrhagic stroke from brain CT scan images (binary classification: Normal vs Stroke).
>
> **Approach:** EfficientNet-B0 with transfer learning, cosine annealing, and emergency alert system.

## 5.1 Dataset & Preprocessing

| Property | Value |
|----------|-------|
| **Total CT scans** | 2,501 |
| **Classes** | 2 (Normal, Stroke) |
| **Split** | Pre-split train/val folders |
| **Augmentation** | Random crop, H-flip, rotation (10°), ColorJitter (no V-flip or hue for CT) |
| **Normalization** | ImageNet mean/std |

> **Note:** CT-specific augmentation choices — we do NOT use vertical flip or hue jitter because these would create medically meaningless images.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PS5 DATASET & MODEL
# ══════════════════════════════════════════════════════════════════════════════

CLASS_TO_LABEL = {"Normal": 0, "Stroke": 1}
LABEL_TO_CLASS = {0: "Normal", 1: "Stroke"}

# CT-specific transforms (no V-flip, no hue jitter)
ps5_train_transform = T.Compose([
    T.Resize((256, 256)),
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

ps5_val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class CTScanDataset(Dataset):
    """PyTorch Dataset for brain CT scan images."""
    def __init__(self, root_dir, split="train", transform=None):
        self.transform = transform
        self.samples = []
        for class_name, label in CLASS_TO_LABEL.items():
            for variant in [class_name, class_name.lower(), class_name.upper()]:
                class_dir = os.path.join(root_dir, "classification", split, variant)
                if os.path.exists(class_dir):
                    for fname in os.listdir(class_dir):
                        if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                            self.samples.append((os.path.join(class_dir, fname), label))
                    break
        print(f"  {split}: {len(self.samples)} images")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)


# ── PS5 Model ────────────────────────────────────────────────────────────────
NUM_CLASSES_PS5 = 2

class StrokeClassifier(nn.Module):
    """
    EfficientNet-B0 fine-tuned for binary stroke detection.

    Input:  CT scan (224×224×3)
    Output: 2 logits [Normal score, Stroke score]
    """
    def __init__(self, num_classes=NUM_CLASSES_PS5, dropout=0.4):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_features = self.backbone.classifier[1].in_features  # 1280
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(p=dropout * 0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.backbone(x)

    def predict_proba(self, x):
        """Returns probability of Stroke (class 1)."""
        logits = self.forward(x)
        return torch.softmax(logits, dim=1)[:, 1]


# ── Model summary ────────────────────────────────────────────────────────────
model_ps5 = StrokeClassifier()
total = sum(p.numel() for p in model_ps5.parameters())
print(f"\nStrokeClassifier (EfficientNet-B0):")
print(f"  Total parameters: {total:,}")

dummy = torch.randn(2, 3, 224, 224)
out = model_ps5(dummy)
prob = model_ps5.predict_proba(dummy)
print(f"  Input:  {dummy.shape}")
print(f"  Output: {out.shape} → 2 class logits")
print(f"  Stroke probabilities: {prob.detach().numpy().round(3)}")

del model_ps5
print("\n✅ PS5 dataset and model defined")

## 5.2 PS5 Training Pipeline

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PS5 TRAINING PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

PS5_CONFIG = {
    "epochs":       25,
    "batch_size":   32,
    "lr":           2e-4,
    "weight_decay": 5e-4,
    "patience":     7,
    "grad_clip":    1.0,
    "scheduler":    "CosineAnnealingLR",
    "loss":         "CrossEntropyLoss",
}

print("PS5 Training Configuration:")
print("-" * 40)
for k, v in PS5_CONFIG.items():
    print(f"  {k:20s}: {v}")


def train_ps5(root_dir, batch_size=32, epochs=25, patience=7):
    """Full PS5 training loop with AUROC-based model selection."""

    print("Loading PS5 CT scan dataset...")
    train_dataset = CTScanDataset(root_dir, split="train", transform=ps5_train_transform)
    val_dataset   = CTScanDataset(root_dir, split="val",   transform=ps5_val_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                              num_workers=0, pin_memory=True)

    model = StrokeClassifier().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=PS5_CONFIG["lr"],
                                   weight_decay=PS5_CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    save_path = os.path.join(PS2_MODEL_DIR, "best_ps5_classifier.pt")
    best_auroc, patience_cnt = 0.0, 0
    history = {"train_loss": [], "val_acc": [], "val_auroc": []}

    print(f"\n{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Acc':>7} | {'AUROC':>6} | {'Note':>12}")
    print("-" * 60)

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc  = correct / total * 100

        # Validate
        model.eval()
        val_correct, val_total = 0, 0
        all_preds, all_labels, all_probs = [], [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                preds = outputs.argmax(dim=1)
                probs = torch.softmax(outputs, dim=1)[:, 1]
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

        val_acc = val_correct / val_total * 100
        auroc   = roc_auc_score(all_labels, all_probs)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_acc"].append(val_acc)
        history["val_auroc"].append(auroc)

        improved = auroc > best_auroc
        if improved:
            best_auroc = auroc
            patience_cnt = 0
            torch.save({"epoch": epoch, "model_state": model.state_dict(),
                        "val_acc": val_acc, "auroc": auroc}, save_path)

        note = f"* BEST" if improved else ""
        print(f"{epoch:>5} | {train_loss:>10.4f} | {train_acc:>8.2f}% | {val_acc:>6.2f}% | {auroc:>5.3f} | {note}")

        if not improved:
            patience_cnt += 1
            if patience_cnt >= patience:
                print(f"\nEarly stopping at epoch {epoch}")
                break

    # Final evaluation
    checkpoint = torch.load(save_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(checkpoint["model_state"])

    print(f"\nBest AUROC: {best_auroc:.4f}")
    print(f"\n{classification_report(all_labels, all_preds, target_names=['Normal', 'Stroke'])}")

    return model, history, all_preds, all_labels, all_probs


# ── Run PS5 training (uncomment to execute) ──────────────────────────────────
# model_ps5, hist_ps5, preds_ps5, labels_ps5, probs_ps5 = train_ps5(PS5_DATA_ROOT)

print("\n💡 PS5 training cell ready — uncomment to run")
print("   Reported results: AUROC=0.982 | Accuracy=92.2% | F1(stroke)=0.89")

## 5.4 PS5 Evaluation & Results

| Metric | Value |
|--------|-------|
| **AUROC** | **0.982** |
| **Accuracy** | **92.2%** |
| **F1 (Stroke)** | **0.890** |
| **F1 (Normal)** | **0.940** |
| **Total CT scans** | 2,501 |
| **Best Epoch** | 18 / 25 |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PS5 EVALUATION VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════════

def plot_ps5_evaluation(preds, labels, probs, history=None):
    """Generate evaluation plots for PS5."""

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("PS5: CT Stroke Detector — Evaluation Report", fontsize=15, fontweight="bold")

    # 1. ROC Curve
    fpr, tpr, _ = roc_curve(labels, probs)
    auroc = roc_auc_score(labels, probs)
    axes[0].plot(fpr, tpr, color="#6366F1", lw=2.5, label=f"AUROC = {auroc:.4f}")
    axes[0].plot([0, 1], [0, 1], "k--", lw=0.8)
    axes[0].fill_between(fpr, tpr, alpha=0.1, color="#6366F1")
    axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title("ROC Curve"); axes[0].legend(fontsize=12); axes[0].grid(alpha=0.3)

    # 2. Confusion Matrix
    cm = confusion_matrix(labels, preds)
    im = axes[1].imshow(cm, cmap="PuBu", aspect="auto")
    for i in range(2):
        for j in range(2):
            axes[1].text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=14,
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
    axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["Normal", "Stroke"])
    axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(["Normal", "Stroke"])
    axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")
    axes[1].set_title("Confusion Matrix")

    # 3. Score distribution
    probs_arr = np.array(probs)
    labels_arr = np.array(labels)
    axes[2].hist(probs_arr[labels_arr == 0], bins=30, alpha=0.7, color="#94A3B8", label="Normal", density=True)
    axes[2].hist(probs_arr[labels_arr == 1], bins=30, alpha=0.7, color="#EF4444", label="Stroke", density=True)
    axes[2].axvline(0.5, color="black", ls="--", lw=2, label="Threshold=0.5")
    axes[2].set_xlabel("Stroke Probability"); axes[2].set_ylabel("Density")
    axes[2].set_title("Score Distribution"); axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("outputs/ps5_evaluation.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("📊 Saved to outputs/ps5_evaluation.png")


print("✅ PS5 evaluation functions defined")

---
<a id="6-specialist-recommender-system"></a>
# 6. Specialist Recommender System

A **100% free** specialist recommendation and hospital finder system — **no API keys required**.

### Components:
1. **Diagnosis → Specialist Mapping** — Offline lookup with urgency levels
2. **IP-based Location Detection** — via ip-api.com (free)
3. **Pre-coded Indian Cities** — 120+ cities with instant geocoding
4. **Hospital Search** — Overpass API (OpenStreetMap) for nearby hospitals within 15-30km
5. **Distance Calculation** — Haversine formula for km distances
6. **Rich Results** — Phone, website, opening hours, Google Maps link

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SPECIALIST RECOMMENDER (100% free, no API keys)
# ══════════════════════════════════════════════════════════════════════════════

import requests

SPECIALIST_MAP = {
    "ps2_high":     ["Intensivist / Critical Care", "General Physician", "Cardiologist"],
    "ps2_moderate": ["General Physician", "Internal Medicine Specialist", "Cardiologist"],
    "ps2_low":      ["General Physician"],
    "ps1_grade1":   ["General Physician", "Podiatrist"],
    "ps1_grade2":   ["Podiatrist", "Diabetologist"],
    "ps1_grade3":   ["Podiatrist", "Vascular Surgeon", "Diabetologist"],
    "ps1_grade4":   ["Vascular Surgeon", "Orthopedic Surgeon", "Podiatrist"],
    "ps5_stroke":   ["Neurologist", "Neurosurgeon", "Emergency Physician"],
    "ps5_normal":   ["Neurologist", "General Physician"],
}

URGENCY_MAP = {
    "ps2_high":     ("🔴 URGENT",    "Visit emergency or call an ambulance immediately."),
    "ps2_moderate": ("🟠 Soon",      "Book an appointment within 24 hours."),
    "ps2_low":      ("🟢 Routine",   "Schedule a routine check-up when convenient."),
    "ps1_grade1":   ("🟢 Routine",   "Routine podiatry appointment recommended."),
    "ps1_grade2":   ("🟠 Soon",      "Book a diabetology/podiatry appointment within a week."),
    "ps1_grade3":   ("🔴 URGENT",    "Seek urgent surgical/podiatry review within 24 hours."),
    "ps1_grade4":   ("🔴 EMERGENCY", "Go to emergency immediately — gangrene risk."),
    "ps5_stroke":   ("🔴 EMERGENCY", "Call emergency services NOW. Every minute matters."),
    "ps5_normal":   ("🟢 Routine",   "Follow up with a neurologist if symptoms persist."),
}


def get_specialists_for_diagnosis(diagnosis_key):
    """Returns specialist list + urgency for a given diagnosis key. Fully offline."""
    specialists = SPECIALIST_MAP.get(diagnosis_key, ["General Physician"])
    urgency_level, urgency_msg = URGENCY_MAP.get(diagnosis_key, ("🟢 Routine", "Consult a doctor."))
    return {
        "diagnosis_key":   diagnosis_key,
        "urgency_level":   urgency_level,
        "urgency_message": urgency_msg,
        "specialists":     specialists,
    }


def _haversine(lat1, lon1, lat2, lon2):
    """Distance in km between two lat/lng points."""
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2) ** 2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


# ── Demo: Specialist Recommendations ─────────────────────────────────────────
print("=" * 60)
print("SPECIALIST RECOMMENDER — DEMO")
print("=" * 60)

demo_cases = [
    ("ps2_high",   "Patient deterioration — HIGH risk"),
    ("ps1_grade3", "Diabetic foot wound — Grade 3 (Abscess)"),
    ("ps5_stroke", "CT scan — Stroke DETECTED"),
]

for key, description in demo_cases:
    info = get_specialists_for_diagnosis(key)
    print(f"\n📋 {description}")
    print(f"   Urgency:     {info['urgency_level']} — {info['urgency_message']}")
    print(f"   Specialists: {', '.join(info['specialists'])}")

print("\n✅ Recommender system operational (100% free, no API keys)")

---
<a id="7-streamlit-application"></a>
# 7. Streamlit Application — Live Demo

The SentinAl Streamlit application (`app.py`, ~1,300 lines) provides a complete clinical interface:

### Application Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                       STREAMLIT UI (app.py)                      │
│  ┌───────────────────────────────────────────────────────────┐   │
│  │  Sidebar Navigation                                        │   │
│  │  ├── Overview (metrics dashboard)                          │   │
│  │  ├── PS2: Vital Signs Monitor                              │   │
│  │  ├── PS1: Foot Wound Grader                                │   │
│  │  └── PS5: CT Stroke Detector                               │   │
│  └───────────────────────────────────────────────────────────┘   │
├──────────────────────────────────────────────────────────────────┤
│  Model Inference Layer (@st.cache_resource)                       │
│  ├── TemporalTransformer (PS2 — 434K params)                     │
│  ├── FootWoundClassifier (PS1 — 4.3M params)                     │
│  └── StrokeClassifier    (PS5 — 4.3M params)                     │
├──────────────────────────────────────────────────────────────────┤
│  Support Services                                                 │
│  ├── Ollama LLM (local, qwen2.5:3b) — Clinical explanations      │
│  ├── Specialist Recommender — Diagnosis → specialist + urgency    │
│  └── OpenStreetMap APIs — Nearby hospital search                  │
└──────────────────────────────────────────────────────────────────┘
```

### Key UI Features:
- **Branded loading screen** with shimmer animation
- **6-panel Plotly dashboard** for vital sign trends
- **Color-coded risk alerts** (RED/AMBER/GREEN)
- **CLIP image validation** before wound grading
- **Interactive chat** with local LLM for clinical Q&A
- **Hospital finder** with Google Maps links

### Running the App:
```bash
streamlit run app.py
```

---
<a id="8-conclusions--future-work"></a>
# 8. Conclusions & Future Work

## Summary of Results

| Module | Model | Key Metric | Dataset | Key Innovation |
|--------|-------|-----------|---------|----------------|
| **PS2: Vital Signs** | Temporal Transformer | **AUROC 0.996** | 293K rows, 7K patients | 8-head self-attention + Focal Loss |
| **PS1: Wound Grading** | EfficientNet-B0 | **97.05% Accuracy** | 9,934 images | CLIP validation + weighted loss |
| **PS5: Stroke Detection** | EfficientNet-B0 | **AUROC 0.982** | 2,501 CT scans | Emergency alert system |

## Key Technical Contributions

1. **Unified Platform:** Three distinct clinical AI modules integrated into a single privacy-first application
2. **Edge AI Architecture:** 100% local inference — zero patient data leaves the device (HIPAA/DISHA compliant)
3. **Class Imbalance Solutions:** Focal Loss + WeightedRandomSampler + threshold optimization for 94:6 imbalance
4. **Clinical Feature Engineering:** 34 domain-driven features including qSOFA, Shock Index, MAP, and temporal trends
5. **Transfer Learning:** EfficientNet-B0 pretrained on 1M images, fine-tuned with only 2.5K-10K medical images
6. **CLIP Validation:** Zero-shot image classification prevents garbage-in-garbage-out
7. **Intelligent Referral:** Automated specialist recommendation with urgency mapping and nearby hospital search (100% free APIs)

## Limitations

- PS5 does not include lesion segmentation (mentioned in problem statement — planned for next iteration)
- PS1 uses wound photographs rather than plantar pressure sensor data (adapted to available dataset)
- Threshold optimization is done on validation set — ideally requires a held-out test set
- Ollama LLM requires separate installation for clinical explanations

## Future Roadmap

| Phase | Plans |
|-------|-------|
| **Next** | ECG arrhythmia detection (PS3), X-ray pneumonia classifier (PS4) |
| **Medium-term** | Lesion segmentation for stroke CT, plantar pressure heatmaps, multi-language clinical summaries |
| **Long-term** | Federated learning across hospitals, mobile app for point-of-care, EHR integration (HL7 FHIR) |

---

## References

1. Lin, T-Y et al. (2017). *Focal Loss for Dense Object Detection.* ICCV.
2. Tan, M. & Le, Q. (2019). *EfficientNet: Rethinking Model Scaling for CNNs.* ICML.
3. Vaswani, A. et al. (2017). *Attention Is All You Need.* NeurIPS.
4. Radford, A. et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision.* ICML (CLIP).
5. Wagner, F. W. (1981). *The Dysvascular Foot: A System for Diagnosis and Treatment.* Foot & Ankle.

---

*This notebook serves as the complete technical report for the SentinAl hackathon submission. All code is reproducible — update the data paths and uncomment training cells to re-run experiments.*